In [ ]:
# ============================================================
# EXPORT INDEPENDANT DES VALEURS DE SHAPLEY PAR FOLD
# Aucun classifieur n'est réappris dans cette cellule.
#
# Fichiers créés :
#   fold_01/adult_train_fold_01_shapley.csv
#   fold_01/adult_test_fold_01_shapley.csv
#   ...
#   fold_10/adult_train_fold_10_shapley.csv
#   fold_10/adult_test_fold_10_shapley.csv
#
# Colonnes de chaque fichier :
#   id ; fold ; Class ; shapley_...
# ============================================================

from pathlib import Path
import tempfile

import pandas as pd

from sklearn.model_selection import StratifiedKFold

from khiops import core as kh
from khiops.core.dictionary import read_dictionary_file
from pathlib import Path

In [ ]:
# Paramètres injectables par Papermill
DATA_PATH = "Adult.txt"
TARGET_COL = "class"
OUTPUT_DIR = "khiops_output_adult"
SEP = "\t"
ID_COL = None
N_SPLITS = 10
RANDOM_STATE = 42
# Séparateur des fichiers CSV produits
OUTPUT_SEP = ";"
# Nombre très élevé pour prendre toutes les variables du prédicteur.
MAX_VARIABLE_IMPORTANCES = 10_000

In [ ]:
DATASET_NAME = Path(DATA_PATH).stem
OUTPUT_DIR = Path(OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. VERIFICATION DE KHIOPS
#
# Les fonctionnalités d'interprétation / Shapley nécessitent
# Khiops 11 ou une version plus récente.
# ============================================================

khiops_version = kh.get_khiops_version()
khiops_major_version = int(str(khiops_version).split(".")[0])

if khiops_major_version < 11:
    raise RuntimeError(
        f"Version détectée : Khiops {khiops_version}\n"
        "Le calcul natif des contributions de Shapley nécessite "
        "Khiops 11 ou une version ultérieure."
    )

print(f"Version Khiops détectée : {khiops_version}")


In [ ]:
# ============================================================
# 3. RECONSTRUCTION DES DONNEES ET DES FOLDS
#
# Cette préparation est volontairement identique au notebook
# fourni, afin de retrouver les mêmes lignes train et test.
# ============================================================

df = pd.read_csv(DATA_PATH, sep=SEP)
df.columns = df.columns.str.strip()

if TARGET_COL not in df.columns:
    raise ValueError(
        f"La colonne cible '{TARGET_COL}' est absente.\n"
        f"Colonnes disponibles : {df.columns.tolist()}"
    )

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .replace({"?": pd.NA, "": pd.NA})
    )

df = df.dropna(subset=[TARGET_COL]).copy()

if ID_COL is not None:
    if ID_COL not in df.columns:
        raise ValueError(
            f"La colonne ID_COL '{ID_COL}' est absente.\n"
            f"Colonnes disponibles : {df.columns.tolist()}"
        )

    ids = df[ID_COL].copy()
    df_model = df.drop(columns=[ID_COL]).copy()

else:
    # Même comportement que dans votre notebook.
    # L'identifiant exporté correspond à l'index initial du fichier.
    ids = pd.Series(df.index, index=df.index, name="row_id")
    df_model = df.copy()

X = df_model.drop(columns=[TARGET_COL]).copy()

y_raw = df_model[TARGET_COL].copy()

if pd.api.types.is_numeric_dtype(y_raw):

    y_numeric = pd.to_numeric(y_raw, errors="raise")

    if (y_numeric.dropna() % 1 == 0).all():
        y = y_numeric.astype("int64")
    else:
        y = y_numeric.astype("float64")

else:

    y = pd.Series(
        y_raw
        .astype("string")
        .str.strip()
        .to_numpy(dtype=str),
        index=y_raw.index,
        name=TARGET_COL,
        dtype=object,
    )

y.name = TARGET_COL




for col in X.columns:
    if (
        pd.api.types.is_string_dtype(X[col])
        or isinstance(X[col].dtype, pd.StringDtype)
    ):
        X[col] = X[col].astype(object)

if y.value_counts().min() < N_SPLITS:
    raise ValueError(
        f"La plus petite classe possède {y.value_counts().min()} lignes. "
        f"Impossible de créer {N_SPLITS} folds stratifiés."
    )

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

In [ ]:
# ============================================================
# 4. FONCTIONS UTILITAIRES
# ============================================================

def get_predictor_dictionary_name(model_kdic_path):
    """
    Recherche le dictionnaire Khiops correspondant au classifieur
    déjà appris : il contient une variable de structure 'Classifier'.
    """
    domain = kh.read_dictionary_file(str(model_kdic_path))

    predictor_candidates = []

    for dictionary in domain.dictionaries:
        has_classifier = any(
            variable.type == "Structure"
            and variable.structure_type == "Classifier"
            for variable in dictionary.variables
        )

        if has_classifier:
            predictor_candidates.append(dictionary.name)

    if len(predictor_candidates) != 1:
        raise RuntimeError(
            "Impossible d'identifier de manière unique le dictionnaire "
            f"du classifieur dans :\n{model_kdic_path}\n\n"
            f"Dictionnaires détectés comme prédicteurs : "
            f"{predictor_candidates}\n"
            f"Tous les dictionnaires disponibles : "
            f"{[dictionary.name for dictionary in domain.dictionaries]}"
        )

    return predictor_candidates[0]


def find_model_kdic(classifier_dir):
    """
    Recherche le fichier .kdic contenant le modèle appris par
    KhiopsClassifier dans le répertoire classifier du fold.
    """
    kdic_paths = sorted(classifier_dir.rglob("*.kdic"))

    if not kdic_paths:
        raise FileNotFoundError(
            f"Aucun fichier .kdic n'a été trouvé dans :\n{classifier_dir}\n\n"
            "Le répertoire du classifieur doit contenir le modèle Khiops "
            "appris lors de l'exécution du notebook initial."
        )

    valid_models = []

    for kdic_path in kdic_paths:
        try:
            predictor_name = get_predictor_dictionary_name(kdic_path)
            valid_models.append((kdic_path, predictor_name))
        except RuntimeError:
            # Certains .kdic éventuellement présents peuvent être des
            # dictionnaires intermédiaires et non des classifieurs.
            continue

    if len(valid_models) != 1:
        raise RuntimeError(
            f"Impossible d'identifier de manière unique le modèle appris "
            f"dans :\n{classifier_dir}\n\n"
            f"Modèles candidats :\n"
            + "\n".join(
                f" - {path} | dictionnaire : {dictionary_name}"
                for path, dictionary_name in valid_models
            )
            + "\n\nTous les fichiers .kdic trouvés :\n"
            + "\n".join(f" - {path}" for path in kdic_paths)
        )

    return valid_models[0]


def get_interpreter_dictionary_and_shapley_columns(interpreter_kdic_path):
    """
    Recherche :
    - le dictionnaire d'interprétation ;
    - les variables calculées par ContributionAt(...), qui portent
      les contributions de Shapley produites par Khiops.
    """
    domain = kh.read_dictionary_file(str(interpreter_kdic_path))

    interpreter_candidates = []

    for dictionary in domain.dictionaries:
        has_interpreter_structure = any(
            variable.type == "Structure"
            and variable.structure_type == "ClassifierInterpreter"
            for variable in dictionary.variables
        )

        shapley_columns = [
            variable.name
            for variable in dictionary.variables
            if "ContributionAt(" in str(variable.rule)
        ]

        if has_interpreter_structure and shapley_columns:
            interpreter_candidates.append(
                (dictionary.name, shapley_columns)
            )

    if len(interpreter_candidates) != 1:
        raise RuntimeError(
            "Impossible d'identifier le dictionnaire d'interprétation "
            f"dans :\n{interpreter_kdic_path}\n\n"
            f"Candidats trouvés : {interpreter_candidates}\n"
            f"Dictionnaires disponibles : "
            f"{[dictionary.name for dictionary in domain.dictionaries]}"
        )

    return interpreter_candidates[0]


def export_shapley_for_split(
    fold_dir,
    fold_number,
    split_name,
    X_split,
    y_split,
    id_split,
    interpreter_kdic_path,
    interpreter_dictionary_name,
    shapley_columns,
):
    """
    Déploie le dictionnaire d'interprétation sur X_split.

    Le fichier final ne contient que :
      - id ;
      - fold ;
      - variable cible ;
      - colonnes shapley_...
    """

    with tempfile.TemporaryDirectory(
        prefix=f"khiops_shapley_fold_{fold_number:02d}_{split_name}_"
    ) as temporary_directory:

        temporary_directory = Path(temporary_directory)

        input_path = (
            temporary_directory
            / f"{DATASET_NAME}_{split_name}_fold_{fold_number:02d}_input.tsv"
        )

        raw_deployment_path = (
            temporary_directory
            / f"{DATASET_NAME}_{split_name}_fold_{fold_number:02d}_raw.csv"
        )

        # Le déploiement du modèle se fait sur les variables explicatives
        # initiales uniquement. La cible n'est pas fournie à Khiops ici.
        # ------------------------------------------------------------
        # Le dictionnaire d'interprétation contient également la cible
        # TARGET_COL. On l'ajoute donc au fichier temporaire fourni à
        # Khiops afin d'éviter le warning indiquant que cette variable
        # est absente.
        #
        # Il n'y a aucune fuite de données :
        # deploy_model applique seulement un modèle déjà appris ; il ne
        # réapprend pas le modèle et la cible n'est pas utilisée comme
        # variable prédictive.
        # ------------------------------------------------------------
        
        deployment_df = X_split.copy()
        
        # .to_numpy() est important : il évite un alignement accidentel
        # entre les index originaux de X_split et ceux de y_split.
        deployment_df[TARGET_COL] = pd.Series(y_split).to_numpy()
        
        deployment_df.to_csv(
            input_path,
            index=False,
            sep=SEP,
        )

        # IMPORTANT :
        # deploy_model applique un modèle existant.
        # Il ne réalise aucun apprentissage.
        kh.deploy_model(
            str(interpreter_kdic_path),
            interpreter_dictionary_name,
            str(input_path),
            str(raw_deployment_path),
            detect_format=False,
            header_line=True,
            field_separator=SEP,
            output_header_line=True,
            output_field_separator=OUTPUT_SEP,
            log_file_path=str(
                fold_dir
                / (
                    f"{DATASET_NAME}_{split_name}_"
                    f"fold_{fold_number:02d}_shapley_deployment.log"
                )
            ),
        )

        raw_shapley_df = pd.read_csv(
            raw_deployment_path,
            sep=OUTPUT_SEP,
        )

    missing_columns = [
        column
        for column in shapley_columns
        if column not in raw_shapley_df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Des colonnes Shapley sont absentes pour le fold "
            f"{fold_number}, jeu {split_name}.\n\n"
            f"Colonnes attendues absentes : {missing_columns}\n\n"
            f"Colonnes produites par Khiops :\n"
            f"{raw_shapley_df.columns.tolist()}"
        )

    # Conservation exclusive des valeurs Shapley.
    shapley_df = raw_shapley_df[shapley_columns].copy()

    # Préfixe homogène et explicite dans le CSV final.
    shapley_df.columns = [
        f"shapley_{column}"
        for column in shapley_df.columns
    ]

    if len(shapley_df) != len(X_split):
        raise RuntimeError(
            f"Nombre de lignes incohérent pour le fold {fold_number}, "
            f"jeu {split_name} :\n"
            f" - données en entrée : {len(X_split)}\n"
            f" - résultats Shapley : {len(shapley_df)}"
        )

    # Le fichier final ne contient pas les Conditional Info pondérées.
    output_df = pd.concat(
        [
            id_split.rename("id").reset_index(drop=True),
            pd.Series(
                fold_number,
                index=range(len(id_split)),
                name="fold",
            ),
            y_split.rename(TARGET_COL).reset_index(drop=True),
            shapley_df.reset_index(drop=True),
        ],
        axis=1,
    )

    output_path = (
        fold_dir
        / f"{DATASET_NAME}_{split_name}_fold_{fold_number:02d}_shapley.csv"
    )

    output_df.to_csv(
        output_path,
        index=False,
        sep=OUTPUT_SEP,
    )

    return output_path, output_df.columns.tolist()


In [ ]:
# ============================================================
# 5. CREATION DES EXPORTS SHAPLEY POUR LES 10 FOLDS
# ============================================================

created_shapley_files = []

for fold_number, (train_index, test_index) in enumerate(
    cv.split(X, y),
    start=1,
):
    print("\n" + "=" * 75)
    print(f"SHAPLEY - FOLD {fold_number}/{N_SPLITS}")
    print("=" * 75)

    fold_dir = OUTPUT_DIR / f"fold_{fold_number:02d}"
    classifier_dir = fold_dir / "classifier"

    if not fold_dir.exists():
        raise FileNotFoundError(
            f"Le répertoire du fold est absent :\n{fold_dir}"
        )

    if not classifier_dir.exists():
        raise FileNotFoundError(
            f"Le répertoire du classifieur est absent :\n{classifier_dir}"
        )

    # --------------------------------------------------------
    # 5.1. Recherche du modèle existant appris dans le notebook
    # --------------------------------------------------------

    model_kdic_path, predictor_dictionary_name = find_model_kdic(
        classifier_dir
    )

    print(f"Modèle existant réutilisé : {model_kdic_path}")
    print(f"Dictionnaire prédicteur : {predictor_dictionary_name}")

    # --------------------------------------------------------
    # 5.2. Création du dictionnaire d'interprétation Shapley
    #      Classement des variables en importance globale.
    #
    # Cette opération ne relance pas l'apprentissage du classifieur.
    # --------------------------------------------------------

    interpreter_dir = fold_dir / "shapley_interpreter"
    interpreter_dir.mkdir(parents=True, exist_ok=True)

    interpreter_kdic_path = (
        interpreter_dir
        / f"{DATASET_NAME}_fold_{fold_number:02d}_global_shapley.kdic"
    )

    kh.interpret_predictor(
        str(model_kdic_path),
        predictor_dictionary_name,
        str(interpreter_kdic_path),
        max_variable_importances=MAX_VARIABLE_IMPORTANCES,
        importance_ranking="Global",
        log_file_path=str(
            interpreter_dir
            / f"{DATASET_NAME}_fold_{fold_number:02d}_global_shapley.log"
        ),
    )

    (
        interpreter_dictionary_name,
        shapley_columns,
    ) = get_interpreter_dictionary_and_shapley_columns(
        interpreter_kdic_path
    )

    print(
        f"Colonnes de valeurs Shapley détectées : "
        f"{len(shapley_columns)}"
    )

    # --------------------------------------------------------
    # 5.3. Reconstruction des données train/test de ce fold
    # --------------------------------------------------------

    X_train = X.iloc[train_index].copy()
    X_test = X.iloc[test_index].copy()

    y_train = y.iloc[train_index].copy()
    y_test = y.iloc[test_index].copy()

    id_train = ids.iloc[train_index].copy()
    id_test = ids.iloc[test_index].copy()

    # --------------------------------------------------------
    # 5.4. Déploiement du même modèle sur train et test
    # --------------------------------------------------------

    train_output_path, train_columns = export_shapley_for_split(
        fold_dir=fold_dir,
        fold_number=fold_number,
        split_name="train",
        X_split=X_train,
        y_split=y_train,
        id_split=id_train,
        interpreter_kdic_path=interpreter_kdic_path,
        interpreter_dictionary_name=interpreter_dictionary_name,
        shapley_columns=shapley_columns,
    )

    test_output_path, test_columns = export_shapley_for_split(
        fold_dir=fold_dir,
        fold_number=fold_number,
        split_name="test",
        X_split=X_test,
        y_split=y_test,
        id_split=id_test,
        interpreter_kdic_path=interpreter_kdic_path,
        interpreter_dictionary_name=interpreter_dictionary_name,
        shapley_columns=shapley_columns,
    )

    created_shapley_files.extend(
        [
            train_output_path,
            test_output_path,
        ]
    )

    print("\nNouveaux fichiers Shapley créés :")
    print(f" - {train_output_path}")
    print(f" - {test_output_path}")

    print("\nColonnes exportées :")
    print("\n".join(f" - {column}" for column in train_columns))


In [ ]:
# ============================================================
# 6. SYNTHESE
# ============================================================

print("\n" + "=" * 75)
print("EXPORT DES VALEURS DE SHAPLEY TERMINE")
print("=" * 75)

print("\nFichiers créés :")

for output_path in created_shapley_files:
    print(f" - {output_path}")